# LeetCode 3581 — Count Odd Letters from Number bằng Python

Link đề: https://leetcode.com/problems/count-odd-letters-from-number/ · Độ khó: **Easy** · Chủ đề: Hash Table, String, Bit Manipulation, Counting

Skeleton LeetCode:

```python
class Solution:
    def countOddLetters(self, n: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đổi **từng chữ số** của `n` thành **tên tiếng Anh** (`4 -> "four"`), **nối** các từ theo thứ tự từ trái sang phải thành một chuỗi, rồi đếm **có bao nhiêu chữ cái khác nhau xuất hiện một số lần LẺ**.
2. Dùng `Counter`/mảng đếm 26 ô để đếm tần suất chữ cái, và hiểu "xuất hiện lẻ" = tần suất `% 2 == 1`.
3. Biết kỹ thuật **bitmask + XOR**: mỗi chữ cái là một bit; gặp chữ cái nào thì **lật** bit đó (`mask ^= 1 << idx`); cuối cùng đếm số bit `1` (popcount).
4. Viết 4 cách: ghép chuỗi + `Counter`, mảng parity 26 ô, bitmask với mặt nạ đã tính sẵn cho từng chữ số, và đếm tần suất chữ số trước rồi mới nhân với số chữ cái; đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: đếm chữ cái **khác nhau** thay vì **lẻ**, đếm chữ cái **chẵn**, cộng **số lần** thay vì **số chữ cái**, và quên chữ số `0` là `"zero"`.

## 2. Đề bài & ví dụ

Cho số nguyên dương `n`. Thực hiện:

1. Đổi **mỗi chữ số** của `n` thành **tên tiếng Anh** viết thường (`0 -> "zero"`, `1 -> "one"`, ..., `9 -> "nine"`).
2. **Nối** các từ theo thứ tự các chữ số của `n` (trái sang phải) thành một chuỗi `s`.
3. Trả về **số chữ cái khác nhau** trong `s` mà **xuất hiện một số lần lẻ**.

| Ví dụ | n | Output | Giải thích |
|---|---|---|---|
| 1 | `41` | 5 | `s = "fourone"`: `f1 o2 u1 r1 n1 e1` -> lẻ: `f, u, r, n, e` = 5 chữ cái |
| 2 | `20` | 5 | `s = "twozero"`: `t1 w1 o2 z1 e1 r1` -> lẻ: `t, w, z, e, r` = 5 chữ cái |

Constraints: `1 <= n <= 10^9`.

In [1]:
WORDS = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine"]
n = 41
s = "".join(WORDS[int(ch)] for ch in str(n))
print("s =", s)
counts = {}
for ch in s:
    counts[ch] = counts.get(ch, 0) + 1
print(counts)
print("chữ cái xuất hiện lẻ:", sorted(c for c, v in counts.items() if v % 2 == 1))
print("đáp án =", sum(1 for v in counts.values() if v % 2 == 1))

s = fourone
{'f': 1, 'o': 2, 'u': 1, 'r': 1, 'n': 1, 'e': 1}
chữ cái xuất hiện lẻ: ['e', 'f', 'n', 'r', 'u']
đáp án = 5


## 3. Kiến thức nền

### 3.1. Danh sách tên số và tra theo chỉ số

Lưu 10 tên số trong một danh sách `WORDS`, chỉ số `d` ứng với chữ số `d`: `WORDS[4] = "four"`. Tách chữ số bằng `str(n)` rồi `int(ch)` để làm chỉ số. Chú ý chữ số `0` có tên `"zero"` (đừng bỏ sót).

In [2]:
print(WORDS[4], WORDS[0], WORDS[9])
print([WORDS[int(ch)] for ch in "20"])

four zero nine
['two', 'zero']


### 3.2. `Counter` và tính chẵn/lẻ của tần suất

`Counter(s)` đếm tần suất từng chữ cái. "Xuất hiện số lần lẻ" nghĩa là `count % 2 == 1`. Nhớ phân biệt: số chữ cái **khác nhau** (`len(counter)`) khác số chữ cái có tần suất **lẻ**.

In [3]:
from collections import Counter
cnt = Counter("fourone")
print(cnt)
print("khác nhau:", len(cnt), "| lẻ:", sum(1 for v in cnt.values() if v % 2 == 1))

Counter({'o': 2, 'f': 1, 'u': 1, 'r': 1, 'n': 1, 'e': 1})
khác nhau: 6 | lẻ: 5


### 3.3. Mảng đếm 26 ô theo chữ cái

Chữ cái `ch` có chỉ số `ord(ch) - ord('a')` trong `0..25`. Mảng `parity = [0] * 26` có thể lưu "lẻ hay chẵn" bằng cách lật `parity[i] ^= 1` mỗi lần gặp chữ cái (XOR với `1`: `0 -> 1`, `1 -> 0`). Cuối cùng `sum(parity)` là số chữ cái lẻ.

In [4]:
print(ord("a"), ord("f") - ord("a"), ord("z") - ord("a"))
parity = [0] * 26
for ch in "fourone":
    parity[ord(ch) - ord("a")] ^= 1
print([chr(i + ord("a")) for i in range(26) if parity[i]], sum(parity))

97 5 25
['e', 'f', 'n', 'r', 'u'] 5


### 3.4. Bitmask, XOR và popcount

Thay mảng 26 ô bằng **một số nguyên** `mask` có 26 bit: bit `i` là parity của chữ cái thứ `i`. Gặp chữ cái: `mask ^= 1 << i` (**XOR** lật bit). Đếm số bit `1`: `bin(mask).count("1")` hoặc `mask.bit_count()` (Python >= 3.10). XOR có tính **giao hoán, kết hợp** và `x ^ x = 0`, nên thứ tự không quan trọng và hai lần gặp cùng một chữ cái triệt tiêu nhau.

In [5]:
mask = 0
for ch in "fourone":
    mask ^= 1 << (ord(ch) - ord("a"))
print(bin(mask), bin(mask).count("1"))
print(5 ^ 5, 5 ^ 0, (5 ^ 3) ^ 3)

0b100100010000000110000 5
0 5 5


### 3.5. Mặt nạ tính sẵn cho từng chữ số

Mỗi chữ số `d` luôn cho cùng một từ, nên có thể **tính trước** `MASK[d]` = bitmask parity của riêng từ `WORDS[d]`. Parity của cả chuỗi là **XOR** các `MASK[d]` theo từng chữ số của `n` (vì XOR kết hợp). Cũng có thể gom **các chữ số lặp**: một chữ số xuất hiện số lần chẵn thì triệt tiêu hoàn toàn.

In [6]:
MASK = []
for w in WORDS:
    m = 0
    for ch in w:
        m ^= 1 << (ord(ch) - ord("a"))
    MASK.append(m)
print([bin(m).count("1") for m in MASK])
r = 0
for ch in "41":
    r ^= MASK[int(ch)]
print(bin(r).count("1"))

[4, 3, 3, 3, 4, 4, 3, 3, 5, 2]
5


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Làm tay với `n = 41`:

```
4 -> "four", 1 -> "one"  =>  s = "fourone"
đếm: f=1, o=2, u=1, r=1, n=1, e=1
lẻ : f, u, r, n, e  (o có 2 lần -> chẵn)
đáp án = 5
```

Ví dụ `n = 111`: `s = "oneoneone"` -> `o=3, n=3, e=3`, cả ba đều lẻ -> `3` (đếm **chữ cái**, không phải tổng số lần `9`). Và `n = 11`: `s = "oneone"` -> mỗi chữ `2` lần -> `0`.

Chỉ cần biết **parity** (chẵn/lẻ) của mỗi chữ cái, không cần số lần chính xác.

## 5. Cách 1 — Brute Force / baseline

Làm đúng như đề: nối các từ thành chuỗi `s`, đếm bằng `Counter`, đếm số chữ cái có tần suất lẻ.

Nút thắt: tạo chuỗi `s` dài tối đa `10` chữ số × `5` ký tự ≈ `50` ký tự — rất nhỏ nên không phải vấn đề thực tế, nhưng cách này lưu cả chuỗi và cả dictionary, trong khi ta chỉ cần **1 bit parity** cho mỗi chữ cái.

In [7]:
from collections import Counter

WORDS = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine"]

class SolutionCounter:
    def countOddLetters(self, n):
        s = "".join(WORDS[int(ch)] for ch in str(n))
        cnt = Counter(s)
        return sum(1 for v in cnt.values() if v % 2 == 1)

print(SolutionCounter().countOddLetters(41))
print(SolutionCounter().countOddLetters(20))

5
5


**Độ phức tạp:** Time `O(L)` với `L` = độ dài chuỗi `s` (tối đa ~50), tức `O(log n)` theo `n`; Space `O(L)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** để biết một chữ cái xuất hiện lẻ hay chẵn, chỉ cần **lật một bit** mỗi lần gặp nó. Không cần lưu tần suất, không cần lưu chuỗi: chạy qua các chữ cái của từng từ rồi lật bit tương ứng.

**Quy tắc "nếu… thì…":**

- Mỗi khi gặp chữ cái `c` -> `parity[c] ^= 1` (hoặc `mask ^= 1 << c`).
- Sau khi duyệt hết -> đáp án là **số bit `1`** (số chữ cái có parity lẻ).

Invariant: tại mọi thời điểm, bit `c` bằng `1` khi và chỉ khi `c` đã xuất hiện một số **lẻ** lần trong phần đã duyệt.

In [8]:
def bit_method(n):
    mask = 0
    for ch in str(n):
        for letter in WORDS[int(ch)]:
            mask ^= 1 << (ord(letter) - ord("a"))
    return bin(mask).count("1")

print(bit_method(41), bit_method(20), bit_method(111), bit_method(11))

5 5 3 0


## 7. Các cách tiếp theo

Ta có 4 cách: (1) ghép chuỗi + `Counter`; (2) mảng parity 26 ô; (3) bitmask XOR với mặt nạ tính sẵn cho từng chữ số; (4) đếm tần suất **chữ số** rồi suy ra parity chữ cái.

### Cách 2 — Mảng parity 26 ô

`parity = [0] * 26`; duyệt từng chữ cái của từng từ, `parity[idx] ^= 1`; trả về `sum(parity)`.

In [9]:
class SolutionParityArray:
    def countOddLetters(self, n):
        parity = [0] * 26
        for ch in str(n):
            for letter in WORDS[int(ch)]:
                parity[ord(letter) - ord("a")] ^= 1
        return sum(parity)

print(SolutionParityArray().countOddLetters(41))
print(SolutionParityArray().countOddLetters(20))

5
5


Độ phức tạp Cách 2: Time `O(L)`, Space `O(26) = O(1)`.

### Cách 3 — Bitmask XOR với mặt nạ tính sẵn

Tính trước `MASK[d]` cho từng chữ số `d` (parity của riêng từ `WORDS[d]`). Với `n`, XOR `MASK[d]` cho từng chữ số. Mỗi chữ số chỉ tốn **một** phép XOR thay vì duyệt từng chữ cái của từ.

In [10]:
class SolutionMask:
    MASK = None

    def __init__(self):
        if SolutionMask.MASK is None:
            masks = []
            for w in WORDS:
                m = 0
                for letter in w:
                    m ^= 1 << (ord(letter) - ord("a"))
                masks.append(m)
            SolutionMask.MASK = masks

    def countOddLetters(self, n):
        mask = 0
        while n > 0:
            mask ^= self.MASK[n % 10]
            n //= 10
        return bin(mask).count("1")

print(SolutionMask().countOddLetters(41))
print(SolutionMask().countOddLetters(20))

5
5


Độ phức tạp Cách 3: Time `O(log n)` (số chữ số, mỗi chữ số một XOR), Space `O(1)` (bảng 10 mặt nạ cố định).

### Cách 4 — Đếm tần suất chữ số trước

Vì thứ tự không quan trọng, ta đếm `dc[d]` = số lần chữ số `d` xuất hiện. Từ `d` xuất hiện `dc[d]` lần đóng góp cho mỗi chữ cái `c` của `WORDS[d]` thêm `dc[d] * (số lần c trong WORDS[d])` lần. Tính tổng tần suất mỗi chữ cái rồi đếm các tổng lẻ.

In [11]:
class SolutionDigitFreq:
    def countOddLetters(self, n):
        dc = [0] * 10
        for ch in str(n):
            dc[int(ch)] += 1
        total = [0] * 26
        for d in range(10):
            if dc[d] == 0:
                continue
            for letter in WORDS[d]:
                total[ord(letter) - ord("a")] += dc[d]
        return sum(1 for v in total if v % 2 == 1)

print(SolutionDigitFreq().countOddLetters(41))
print(SolutionDigitFreq().countOddLetters(20))

5
5


Độ phức tạp Cách 4: Time `O(log n + 10 * 5)`, Space `O(1)`.

## 8. TRACE từng bước

Trace Cách 3 (bitmask) với `n = 41`, chỉ in các chữ cái có bit bật (từ phải sang trái: `1` rồi `4`):

| bước | chữ số | từ | XOR `MASK[d]` | chữ cái lẻ sau bước |
|---|---|---|---|---|
| 1 | 1 | `one` | lật `o, n, e` | `e, n, o` |
| 2 | 4 | `four` | lật `f, o, u, r` | `e, f, n, r, u` (`o` bị lật lại thành chẵn) |

Kết quả: `e, f, n, r, u` -> `5` chữ cái.

In [12]:
def trace_mask(n):
    mask = 0
    print("n =", n)
    for step, ch in enumerate(reversed(str(n)), start=1):
        d = int(ch)
        m = 0
        for letter in WORDS[d]:
            m ^= 1 << (ord(letter) - ord("a"))
        mask ^= m
        odd = [chr(i + ord("a")) for i in range(26) if mask >> i & 1]
        print(f"bước {step}: chữ số {d} ({WORDS[d]}) | chữ cái lẻ = {odd}")
    print("đáp án =", bin(mask).count("1"))
    return bin(mask).count("1")

trace_mask(41)
trace_mask(111)

n = 41
bước 1: chữ số 1 (one) | chữ cái lẻ = ['e', 'n', 'o']
bước 2: chữ số 4 (four) | chữ cái lẻ = ['e', 'f', 'n', 'r', 'u']
đáp án = 5
n = 111
bước 1: chữ số 1 (one) | chữ cái lẻ = ['e', 'n', 'o']
bước 2: chữ số 1 (one) | chữ cái lẻ = []
bước 3: chữ số 1 (one) | chữ cái lẻ = ['e', 'n', 'o']
đáp án = 3


3

## 9. Code nộp LeetCode

In [13]:
from typing import List

class Solution:
    def countOddLetters(self, n: int) -> int:
        words = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine"]
        mask = 0
        for ch in str(n):
            for letter in words[int(ch)]:
                mask ^= 1 << (ord(letter) - ord("a"))
        return bin(mask).count("1")

print(Solution().countOddLetters(41), Solution().countOddLetters(20))

5 5


## 10. Vì sao thuật toán đúng

- **Parity quyết định kết quả:** một chữ cái xuất hiện lẻ lần khi và chỉ khi tần suất `% 2 = 1`. Vậy chỉ cần theo dõi parity của từng chữ cái, không cần tần suất chính xác.
- **XOR đúng là cộng modulo 2:** `x ^ 1` lật bit; sau `t` lần lật, bit bằng `t mod 2`. Do đó bit `c` bằng `1` đúng khi `c` xuất hiện lẻ lần (Cách 2, 3).
- **Thứ tự không quan trọng:** XOR giao hoán và kết hợp nên ta có thể XOR theo từng từ hoặc từng chữ số, theo thứ tự nào cũng được. Do đó mặt nạ tính sẵn `MASK[d]` (parity của một từ) kết hợp bằng XOR đúng bằng parity của cả chuỗi nối (Cách 3).
- **Cách 4:** chữ cái `c` xuất hiện trong chuỗi `sum_d dc[d] * occ(c, WORDS[d])` lần; ta kiểm tra tổng này lẻ — đúng định nghĩa.
- **Đếm chữ cái:** popcount đếm số bit `1`, mỗi bit là một chữ cái khác nhau, nên kết quả là số **chữ cái** (không phải số lần), đúng yêu cầu.
- **Chữ số `0`:** `WORDS[0] = "zero"` được tính như mọi chữ số khác (ví dụ `20 -> "twozero"`).

## 11. Phân tích Time / Space Complexity

Gọi `D = số chữ số của n` (`D <= 10`, tức `D = O(log n)`); mỗi từ có tối đa `5` chữ cái nên `L <= 5 D`.

| Cách | Time | Space |
|---|---|---|
| 1. Ghép chuỗi + `Counter` | `O(L)` | `O(L)` |
| 2. Mảng parity 26 ô | `O(L)` | `O(26) = O(1)` |
| 3. Bitmask + mặt nạ tính sẵn | `O(D)` | `O(1)` |
| 4. Tần suất chữ số | `O(D + 50)` | `O(1)` |

Tất cả đều chạy tức thì (`n <= 10^9`). Cách 3 là gọn và nhanh nhất; cách 2 dễ đọc nhất.

## 12. Bộ test

Test 1: ví dụ đề và edge case (một chữ số, chữ số lặp, có `0`, hai chữ số giống nhau triệt tiêu, `n = 10^9`).

In [14]:
impls = [SolutionCounter(), SolutionParityArray(), SolutionMask(), SolutionDigitFreq(), Solution()]
tests = [
    (41, 5),
    (20, 5),
    (1, 3),                    # "one": o, n, e đều 1 lần
    (2, 3),                    # "two": t, w, o
    (3, 3),                    # "three": t1 h1 r1 e2 -> lẻ: t, h, r
    (7, 3),                    # "seven": s1 e2 v1 n1 -> lẻ: s, v, n
    (11, 0),                   # "oneone": mỗi chữ cái 2 lần
    (111, 3),                  # "oneoneone": o3 n3 e3
    (10, 3),                   # "onezero": o2 n1 e2 z1 r1 -> lẻ: n, z, r
    (88, 0),                   # "eighteight": mỗi chữ cái chẵn lần
    (66, 0),                   # "sixsix"
    (10**9, 3),                # "one" + 9 lần "zero": z9 e10 r9 o10 n1 -> lẻ: z, r, n
    (999999999, 2),            # "nine" x9: n18 i9 e9 -> lẻ: i, e
    (123456789, 6),            # đủ 9 chữ số khác nhau
]

def oracle(n):
    words = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine"]
    s = "".join(words[int(ch)] for ch in str(n))
    return len([c for c in set(s) if s.count(c) % 2 == 1])

for n, exp in tests:
    assert 1 <= n <= 10**9
    assert oracle(n) == exp, (n, oracle(n), exp)
    for im in impls:
        assert im.countOddLetters(n) == exp, (type(im).__name__, n, im.countOddLetters(n), exp)
print("tất cả test cố định OK:", len(tests))

tất cả test cố định OK: 14


Test 2: ngẫu nhiên có seed, đúng constraints (`1 <= n <= 10^9`). Oracle độc lập: `set(s)` và `s.count(c)` (không dùng `Counter`, không dùng XOR).

In [15]:
import random
random.seed(3581)

for t in range(3000):
    kind = random.choice(["small", "mid", "big", "repeat"])
    if kind == "small":
        n = random.randint(1, 99)
    elif kind == "mid":
        n = random.randint(1, 10**5)
    elif kind == "big":
        n = random.randint(1, 10**9)
    else:
        d = random.choice("0123456789")
        n = int((d if d != "0" else "1") + "".join(random.choice([d, "7"]) for _ in range(random.randint(0, 8))))
    assert 1 <= n <= 10**9
    exp = oracle(n)
    for im in impls:
        assert im.countOddLetters(n) == exp, (type(im).__name__, n)
print("3000 test ngẫu nhiên OK")

3000 test ngẫu nhiên OK


Test 3: tính chất — `0 <= đáp án <= 26`; đáp án không đổi khi **hoán vị** các chữ số (bỏ qua số bắt đầu bằng `0`); ghép **hai bản** của cùng một dãy chữ số cho `0` (mỗi chữ cái xuất hiện số lần chẵn); và `n` có một chữ số `d` cho đúng số chữ cái lẻ của từ `WORDS[d]`.

In [16]:
random.seed(581)
for _ in range(1000):
    n = random.randint(1, 10**4)
    digs = list(str(n))
    ans = Solution().countOddLetters(n)
    assert 0 <= ans <= 26
    random.shuffle(digs)
    if digs[0] != "0":
        assert Solution().countOddLetters(int("".join(digs))) == ans
    assert Solution().countOddLetters(int(str(n) + str(n))) == 0
for d in range(1, 10):
    w = WORDS[d]
    assert Solution().countOddLetters(d) == len([c for c in set(w) if w.count(c) % 2 == 1])
print("tính chất (0..26, hoán vị, nhân đôi dãy -> 0, một chữ số): OK")

tính chất (0..26, hoán vị, nhân đôi dãy -> 0, một chữ số): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: đếm chữ cái **khác nhau** thay vì chữ cái xuất hiện **lẻ**

In [17]:
def wrong_distinct(n):
    s = "".join(WORDS[int(ch)] for ch in str(n))
    return len(set(s))                          # SAI: đếm số chữ cái khác nhau, không xét chẵn/lẻ

print("sai :", wrong_distinct(41))
print("đúng:", Solution().countOddLetters(41))
assert wrong_distinct(41) != Solution().countOddLetters(41)

sai : 6
đúng: 5


Vì sao sai: `"fourone"` có `6` chữ cái khác nhau (`f,o,u,r,n,e`) nhưng `o` xuất hiện `2` lần (chẵn) nên chỉ `5` chữ cái lẻ. **Sửa:** chỉ đếm chữ cái có tần suất `% 2 == 1`.

### Lỗi 2: đếm chữ cái xuất hiện số lần **chẵn**

In [18]:
def wrong_even(n):
    s = "".join(WORDS[int(ch)] for ch in str(n))
    return sum(1 for c in set(s) if s.count(c) % 2 == 0)     # SAI: đếm chẵn, đề hỏi lẻ

print("sai :", wrong_even(41))
print("đúng:", Solution().countOddLetters(41))
assert wrong_even(41) != Solution().countOddLetters(41)

sai : 1
đúng: 5


Vì sao sai: ở `"fourone"` chỉ `o` có số lần chẵn nên trả `1`, trong khi đáp án đúng là `5`. Đọc kỹ "**odd** number of times".

### Lỗi 3: cộng **số lần** thay vì đếm **số chữ cái**

In [19]:
def wrong_sum_counts(n):
    s = "".join(WORDS[int(ch)] for ch in str(n))
    return sum(s.count(c) for c in set(s) if s.count(c) % 2 == 1)   # SAI: cộng số lần xuất hiện

print("sai :", wrong_sum_counts(111))
print("đúng:", Solution().countOddLetters(111))
assert wrong_sum_counts(111) != Solution().countOddLetters(111)

sai : 9
đúng: 3


Vì sao sai: `"oneoneone"` có `o`, `n`, `e` mỗi chữ cái `3` lần: cộng số lần cho `9`, còn đề hỏi **bao nhiêu chữ cái** (`3`). **Sửa:** mỗi chữ cái lẻ chỉ cộng `1`.

### Lỗi 4: quên chữ số `0` là `"zero"`

In [20]:
WORDS_NO_ZERO = ["", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine"]    # SAI: thiếu "zero"

def wrong_missing_zero(n):
    s = "".join(WORDS_NO_ZERO[int(ch)] for ch in str(n))
    return len([c for c in set(s) if s.count(c) % 2 == 1])

print("sai :", wrong_missing_zero(20))
print("đúng:", Solution().countOddLetters(20))
assert wrong_missing_zero(20) != Solution().countOddLetters(20)

sai : 3
đúng: 5


Vì sao sai: với `n = 20` chuỗi đúng là `"twozero"` (`5` chữ cái lẻ); bỏ `"zero"` còn `"two"` với `3`. Chữ số `0` vẫn là một chữ số và có tên `"zero"`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm chữ cái xuất hiện **số lần lẻ**" -> **bitmask XOR** hoặc mảng parity: chỉ cần 1 bit mỗi chữ cái.
- XOR hai lần cùng một giá trị triệt tiêu: `x ^ x = 0`. Đây là nguyên lý của nhiều bài (`136. Single Number`, `1371`, palindrome permutation).
- Đếm số bit `1`: `bin(mask).count("1")` hoặc `mask.bit_count()`.
- Nếu thao tác chỉ phụ thuộc **tập hợp/số lần** của các thành phần, có thể **tính trước** cho từng thành phần (mặt nạ cho từng chữ số) rồi gộp.
- Phân biệt: "số chữ cái **khác nhau**" vs "số chữ cái **lẻ**" vs "tổng số lần".
- Đừng quên chữ số `0 -> "zero"`.

## 15. Biến thể của bài

1. Đếm chữ cái xuất hiện số lần **chẵn** (và có mặt trong chuỗi).
2. Dùng tên chữ số ở ngôn ngữ khác hoặc **tên tiếng Anh của cả số** ("one hundred twenty-three").
3. Tìm chữ cái xuất hiện **nhiều nhất** trong chuỗi nối (dùng `Counter.most_common`).
4. Kiểm tra chuỗi nối có thể **hoán vị thành palindrome** không (số chữ cái lẻ `<= 1`, bài `266`).
5. Đếm trên **mọi** số từ `1` đến `n` (cộng dồn mask hoặc đếm theo chữ số).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Ghép chuỗi + `Counter` | làm đúng đề | `O(L)` | `O(L)` | dễ hiểu, oracle |
| 2. Mảng parity 26 ô | lật `parity[c] ^= 1` | `O(L)` | `O(1)` | **dễ đọc, nên nộp** |
| 3. Bitmask + mặt nạ sẵn | XOR theo từng chữ số | `O(D)` | `O(1)` | gọn, nhanh nhất |
| 4. Tần suất chữ số | đếm chữ số rồi suy chữ cái | `O(D + 50)` | `O(1)` | nhìn thấy cấu trúc "chữ số -> chữ cái" |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `countOddLetters(7)` bằng bao nhiêu? (`"seven"`)

**Bài 2 (tính tay):** `n = 66`: ghép chuỗi, đếm tần suất; đáp án là gì?

**Bài 3 (điền TODO):** hoàn thành `word_mask(word)` trả về bitmask parity của một từ.

**Bài 4 (tự viết lại):** viết `odd_letters_set(n)` trả về **tập** các chữ cái xuất hiện lẻ lần (thay vì số lượng).

**Bài 5 (mini challenge):** viết `can_be_palindrome(n)` — chuỗi nối có thể **sắp xếp lại** thành palindrome không (số chữ cái lẻ `<= 1`)?

In [21]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().countOddLetters(7))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def word_mask(word):
    # TODO: mask = 0; với mỗi chữ cái: mask ^= 1 << (ord(c) - ord('a')); trả về mask
    return -1

3


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** ghép tên các chữ số, đếm từng chữ cái, kể những chữ cái có số lần lẻ.

**Mức 2 (vừa):** Bài 3: vòng lặp với `^=`. Bài 4: duyệt `mask` và `chr(i + ord('a'))` cho các bit bật. Bài 5: dùng `bin(mask).count('1') <= 1` trên mask của cả chuỗi.

**Mức 3 (gần đáp án):** Bài 1: `"seven"`: `s1 e2 v1 n1` -> lẻ: `s, v, n` = `3`. Bài 2: `"sixsix"` -> mỗi chữ cái `2` lần -> `0`.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1: "seven" -> s, v, n
assert Solution().countOddLetters(7) == 3

# Bài 2: "sixsix" -> 0
assert Solution().countOddLetters(66) == 0

# Bài 3
def word_mask(word):
    mask = 0
    for c in word:
        mask ^= 1 << (ord(c) - ord("a"))
    return mask
assert bin(word_mask("one")).count("1") == 3
assert bin(word_mask("seven")).count("1") == 3
assert word_mask("oneone") == 0

# Bài 4
def odd_letters_set(n):
    s = "".join(WORDS[int(ch)] for ch in str(n))
    mask = word_mask(s)
    return {chr(i + ord("a")) for i in range(26) if mask >> i & 1}
assert odd_letters_set(41) == {"f", "u", "r", "n", "e"}
assert odd_letters_set(20) == {"t", "w", "z", "e", "r"}
assert len(odd_letters_set(111)) == Solution().countOddLetters(111)

# Bài 5
def can_be_palindrome(n):
    s = "".join(WORDS[int(ch)] for ch in str(n))
    return bin(word_mask(s)).count("1") <= 1
assert can_be_palindrome(11) is True
assert can_be_palindrome(41) is False
assert can_be_palindrome(66) is True
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đếm ký tự xuất hiện lẻ lần bằng bitmask XOR":**

```python
mask = 0
for ch in text:
    mask ^= 1 << (ord(ch) - ord('a'))   # lật bit của chữ cái
return bin(mask).count('1')             # số chữ cái có parity lẻ
```

**Checklist trước khi submit:**

- [ ] Mỗi chữ số đổi đúng tên, **kể cả** `0 -> "zero"`?
- [ ] Đếm chữ cái **lẻ** (không phải khác nhau, không phải chẵn)?
- [ ] Kết quả là số **chữ cái** (mỗi chữ cái tính `1`), không phải tổng số lần?
- [ ] Dùng XOR (`^=`) để lật bit, không phải `|=` (OR) hay `+=`?
- [ ] Đã thử `n` một chữ số, chữ số lặp (`11`, `111`), có `0` (`20`), và `10^9`?
- [ ] Chữ ký đúng `countOddLetters(self, n: int) -> int`?